# Chapter 06 — The Built-In Tools

**Companion to *Pi Agents*** · [`Pi Agents` chapter 06](https://programmer.ie/books/pi/06-chapter/)

| | |
|---|---|
| Chapter | `06-chapter.md` · weight 6 · `/books/pi/06-chapter/` |
| Notebook | `notebooks/pi/06-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

What tools does a plain Pi session actually have, and **what does
`--tools` remove?**

The chapter has no example directory of its own — it deliberately delegates. What
it does have is a boundary worth checking: registered versus active versus declared
to the model, and the difference between a *narrowed* tool set and a *gated* one.


## What this notebook establishes

- A default session registers **eight** tools and declares **four** to the model. The four that are declared are `read, bash, edit, write` — the `defaultTools` default.
- `--tools read,grep,find,ls` against the shipped binary makes a model request for `bash` come back as a **failed tool result**, and the command never runs. The default set runs the same command and creates the file.
- `defaultTools` in a project file **amends** when every entry is `+name` or `-name`, and **replaces** when any entry is a plain name.
- Each registered tool carries `exposure` and `annotations`, and those are readable from a live session.

## What this notebook does **not** establish

- **No MCP server is configured anywhere in this notebook.** As of 1.0.4 `--tools` does not remove MCP tools unless an entry starts with `mcp__`, so the four-tool result here is a statement about built-ins. Chapter 21 exercises the MCP case with a real in-memory server.
- **Tool descriptions, the codemode output limit and `tools.read()` on an image** are named by the chapter and not exercised here.
- **`-nt` / `-nbt` / `-xt` variants and the reload trap** are explicitly not run, by the chapter's own account.
- **No measurement** of codemode's context or latency effect for any workload.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — a live session's tool inventory, and the shipped binary with `--tools`.
* **Evidence scope:** `in_process_runtime` for the inventory; `shipped_binary` for the narrowing. The provider is scripted.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(6))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/06-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: what a plain session has

Three sets, and the difference between them is the whole chapter:

* **registered** — every tool the runtime knows about;
* **active** — the subset in force for this session;
* **declared** — what the provider is told, which is the active set rendered into
  the system message.

The driver is `notebooks/pi/drivers/ch06-inventory.ts`. It is a real file in this
repository, so you can read it before running it.

In [3]:
INVENTORY = pinb.driver_source("ch06-inventory.ts")
tools = pinb.driver(INVENTORY, label="ch06-inventory", timeout=240)
print(pinb.md_table(
    ["tool", "exposure", "readOnlyHint", "registered", "active", "declared to the model"],
    [[t["name"], t["exposure"], str(t["readOnlyHint"]).lower(), "yes",
      "yes" if t["name"] in tools["default"]["active"] else "-",
      "yes" if t["name"] in tools["default"]["declared"] else "-"]
     for t in tools["default"]["registered"]],
))
print()
print(pinb.table(
    ["session", "registered", "active", "declared"],
    [["default", len(tools["default"]["registered"]), len(tools["default"]["active"]), len(tools["default"]["declared"])],
     ["tools: read,grep,find,ls", len(tools["narrowed"]["registered"]), len(tools["narrowed"]["active"]), len(tools["narrowed"]["declared"])]],
    indent="",
))

| tool | exposure | readOnlyHint | registered | active | declared to the model |
|---|---|---|---|---|---|
| read | direct | none | yes | yes | yes |
| bash | direct | none | yes | yes | yes |
| powershell | direct | none | yes | - | - |
| edit | direct | none | yes | yes | yes |
| write | direct | none | yes | yes | yes |
| grep | direct | none | yes | - | - |
| find | direct | none | yes | - | - |
| ls | direct | none | yes | - | - |

session  registered  active  declared
------------------------  ----------  ------  --------
default  8  4  4
tools: read,grep,find,ls  4  4  4


In [4]:
d, n = tools["default"], tools["narrowed"]
meta = lambda s: {t["name"]: (t["exposure"], t["readOnlyHint"]) for t in s["registered"]}

pinb.show_checks([
    pinb.check("the default declared set is read, bash, edit, write",
               d["declared"] == ["read", "bash", "edit", "write"], ", ".join(d["declared"])),
    pinb.check("a default session registers more tools than it declares",
               len(d["registered"]) > len(d["declared"]),
               f"{len(d['registered'])} registered, {len(d['declared'])} declared"),
    pinb.check("every declared tool is active",
               set(d["declared"]) <= set(d["active"]), "declared is a subset of active"),
    pinb.check("every active tool is registered",
               set(d["active"]) <= set(meta(d)), "active is a subset of registered"),
    pinb.check("narrowing with tools: removes from registration too",
               n["declared"] == ["read", "grep", "find", "ls"] and len(n["registered"]) == 4,
               f"declared {n['declared']}, registered {len(n['registered'])}"),
    pinb.check("narrowing does not change the metadata of a kept tool",
               meta(n) == {k: v for k, v in meta(d).items() if k in meta(n)},
               "exposure and annotations are unchanged for the four kept tools"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------------  --------------------------------------------------------------
PASS  the default declared set is read, bash, edit, write  read, bash, edit, write
PASS  a default session registers more tools than it declares  8 registered, 4 declared
PASS  every declared tool is active  declared is a subset of active
PASS  every active tool is registered  active is a subset of registered
PASS  narrowing with tools: removes from registration too  declared ['read', 'grep', 'find', 'ls'], registered 4
PASS  narrowing does not change the metadata of a kept tool  exposure and annotations are unchanged for the four kept tools

6/6 assertions held.


## Experiment: does a narrowed agent actually refuse?

A registered tool that is inactive is one thing; the question is what a **model
that asks for it anyway** receives. This runs the shipped binary: a scripted model
calls `bash`, and the driver reports both the tool result and whether the command
left any trace on disk.

That second half matters. A failed tool result and a command that ran and failed
look the same in a transcript and are different in the world.

In [5]:
NARROWED = pinb.driver_source("ch06-narrowed.ts")
narrow = pinb.driver(NARROWED, label="ch06-narrowed", timeout=300)
print(pinb.md_table(
    ["--tools", "tools declared to the model", "model asked for", "result", "file on disk?"],
    [[k, ", ".join(v["declaredTools"] or []), v["toolName"] or "-",
      "error" if v["isError"] else ("ok" if v["isError"] is False else "no tool call"),
      "YES" if v["fileCreated"] else "no"] for k, v in narrow.items()],
))

| --tools | tools declared to the model | model asked for | result | file on disk? |
|---|---|---|---|---|
| default | read, bash, edit, write | bash | ok | YES |
| narrowed | read, grep, find, ls | bash | error | no |
| narrowedWithNoMcp | read, grep, find, ls | bash | error | no |


In [6]:
pinb.show_checks([
    pinb.check("with the default set, bash runs and leaves a file",
               narrow["default"]["fileCreated"] and narrow["default"]["isError"] is False,
               "ran.flag exists"),
    pinb.check("--tools read,grep,find,ls does not declare bash",
               "bash" not in (narrow["narrowed"]["declaredTools"] or []),
               ", ".join(narrow["narrowed"]["declaredTools"] or [])),
    pinb.check("a model that asks for a narrowed tool gets an error result",
               narrow["narrowed"]["isError"] is True, "isError: true"),
    pinb.check("and the command never ran",
               not narrow["narrowed"]["fileCreated"], "ran.flag absent"),
    pinb.check("--no-mcp does not change the built-in narrowing",
               narrow["narrowedWithNoMcp"]["isError"] is True and not narrow["narrowedWithNoMcp"]["fileCreated"],
               "same result; there are no MCP servers in this notebook"),
])

== Assertions ==
  assertion  observed
----  ----------------------------------------------------------  ------------------------------------------------------
PASS  with the default set, bash runs and leaves a file  ran.flag exists
PASS  --tools read,grep,find,ls does not declare bash  read, grep, find, ls
PASS  a model that asks for a narrowed tool gets an error result  isError: true
PASS  and the command never ran  ran.flag absent
PASS  --no-mcp does not change the built-in narrowing  same result; there are no MCP servers in this notebook

5/5 assertions held.


## Contrasting case: narrowing by configuration, not by flag

`--tools` is the command line. `defaultTools` is the file, and it has a grammar
that is easy to get wrong: **a project list of only `+name` / `-name` entries
amends** the user's selection; **a list containing any plain name replaces** it
outright. Within one list, plain names form the selection and `+`/`-` then apply
in order.

That is the chapter 6 / chapter 7 shared evidence, and it runs here because
chapter 6 has no example directory of its own.

## The evidence chapter 6 delegates to

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [7]:
PATTERNS = ['ch07-settings/settings.test.ts', 'ch32-headless/headless.test.ts']
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch07-settings/settings.test.ts, ch32-headless/headless.test.ts', show="tools")

$ node --test ch07-settings/settings.test.ts, ch32-headless/headless.test.ts
  PASS  20 passed, 0 failed, 0 skipped  in 3403 ms

  tests matching 'tools':
    ok   defaultTools: a project list of only +name and -name edits the user's selection
    ok   defaultTools: a project list containing a plain name replaces the selection outright
    ok   defaultTools: within one list, plain names form the selection and +/- then apply in order
    ok   defaultTools: -name removes from the user's list
    ok   --tools narrows the agent: a model that asks for bash gets a failed result, and nothing runs


## Your turn: predict, then run

**Predict first.** `defaultTools: ["+grep", "+find", "+ls"]` on top of a user
selection of `["read", "bash"]`. What is the result — and what if the project list
were `["read", "powershell"]` instead?

**Then change one input.** `TOOLS` below is what gets declared to the model. Try
`["bash"]` alone, then `["bash", "powershell"]`.

**Predict the boundary.** Which single entry turns an amendment into a
replacement?

In [8]:
TOOLS = ["bash"]
USER_SELECTION = ["read", "bash"]


def merge_default_tools(user: list[str], project: list[str]) -> list[str]:
    """The documented grammar, restated so the amendment/replace rule is visible.

    This is the rule chapter 6 states, not an implementation of Pi. The
    authoritative check is Pi's own SettingsManager, asserted by
    ch07-settings/settings.test.ts, which ran above.
    """
    if not project:
        return list(user)
    if any(e and e[0] not in "+-" for e in project):
        return list(project)
    out = list(user)
    for entry in project:
        name = entry[1:]
        if entry.startswith("+"):
            if name not in out:
                out.append(name)
        elif name in out:
            out.remove(name)
    return out

for project in (["+grep", "+find", "+ls"], ["read", "powershell"], ["read", "bash", "-bash", "+grep"]):
    print(f"  user {USER_SELECTION} + project {project} -> {merge_default_tools(USER_SELECTION, project)}")
print()
print("predicted declared set for TOOLS =", TOOLS)

import json as _json

again = pinb.driver(INVENTORY, label="ch06-exercise", timeout=240, env={"NB_TOOLS": _json.dumps(TOOLS)})
print("observed declared set:  ", again["narrowed"]["declared"])

assert again["narrowed"]["declared"] == TOOLS, "the declared set is what was asked for, in order"
assert set(TOOLS) <= set(meta(again["narrowed"])), "every declared tool is also registered"
print("\nheld: the declared set is the selection, not a filter over a larger one")

  user ['read', 'bash'] + project ['+grep', '+find', '+ls'] -> ['read', 'bash', 'grep', 'find', 'ls']
  user ['read', 'bash'] + project ['read', 'powershell'] -> ['read', 'powershell']
  user ['read', 'bash'] + project ['read', 'bash', '-bash', '+grep'] -> ['read', 'bash', '-bash', '+grep']

predicted declared set for TOOLS = ['bash']


observed declared set:   ['bash']

held: the declared set is the selection, not a filter over a larger one


## Interpretation: three mechanisms that are easy to confuse

| Mechanism | Where | What it stops | What it does **not** stop |
|---|---|---|---|
| `--tools` / `tools:` | command line / settings | the model being *told* about the tool | nothing — the tool was never given |
| `defaultTools` | a settings file | the same, from configuration | ditto |
| a `tool_call` gate | an extension (chapters 11, 16, 17, 21) | the tool **running** | the model still being offered it |

Only the third is a permission. The first two are menu control. A reader who
treats `--tools` as a safety mechanism has misread which boundary it is — which is
the mistake chapter 42 spends its length correcting.

### What this chapter does not settle

* **MCP tools and `--tools` since 1.0.4.** The narrowing above is a statement about
  built-ins; `--tools` keeps MCP tools unless an entry starts with `mcp__`. Chapter
  21 runs that case with a real in-memory MCP server.
* **Codemode.** Named, version-marked (1.0.1 for the output limit, 1.0.3 for
  `image()` writing to a temp file, 1.0.4 for `tools.read()` resolving to an image
  block), and not exercised here.

## Sources, execution mode and limitations

**Execution mode:** **Run** — a live session's tool inventory, three runs of the shipped binary with and without `--tools`, and the two test files this chapter delegates to.

**Evidence scope:** `in_process_runtime` plus `shipped_binary`. This chapter has no example directory of its own, so it has no ledger rows; that is why the two borrowed test files are named explicitly above.

### What was read or run

- **Ran** `makeSession` twice and read `getAllTools()`, `getActiveToolNames()`, `getCallableToolNames()` and the system message the provider received (`drivers/ch06-inventory.ts`).
- **Ran** the shipped binary three times, offline, each in a fresh temporary working directory, checking both the tool result and the filesystem (`drivers/ch06-narrowed.ts`).
- **Ran** `ch07-settings/settings.test.ts` and `ch32-headless/headless.test.ts`.
- **Read** `examples/evidence.json`: chapter 6 has **no rows**, which is consistent with the chapter delegating its evidence.

### Limitations

- **No MCP server exists in this notebook**, so `--tools` behaves as a built-in filter here. The 1.0.4 rule that `--tools` no longer removes MCP tools is therefore *not* exercised; chapter 21 is where it is.
- **The amendment/replace grammar shown in the exercise is the chapter's rule restated**, not an implementation. The authoritative check is the `SettingsManager` assertions in `settings.test.ts`, which ran above.
- **Tool metadata is reported, not trusted.** `annotations.readOnlyHint` is a hint the tool declares about itself; chapter 21's annotation experiment is the demonstration that a server can lie about it.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
